# 第 08 课｜向量、点积与矩阵形状

**学习目标：**用 Python 列表实现向量运算；看懂矩阵的行、列和形状；亲手算出一个简化的 Transformer `QKᵀ / √dₖ` 分数矩阵。

- **预计用时：**2–3 小时（包括练习）。
- **先修知识：**第 01–07 课的列表、循环、函数、异常和 `assert`。
- **环境：**Python 3 + Jupyter；**不需要 NumPy、PyTorch 或显卡**。
- **学习方式：**先预测输出 → 运行单元 → 改一个数 → 解释变化 → 完成练习，再看答案。

> 本课的向量只是我们手工指定的数字。真实大模型的 token embedding 通常是通过训练得到的参数；这里先学计算规则，不把这些数字当作真实语言的语义表示。

## 1. 向量：一组有顺序的数字

今天用一个 Python **列表**表示一个向量。例如 `a = [1, 2, 3]` 是三维向量，因为它有三个分量。

你要区分：**向量的维度 = 分量个数**；向量的某个分量值大，不代表它的维度更高。

In [1]:
a = [1, 2, 3]
b = [-1, 0, 2]
print("向量 a：", a)
print("a 的维度：", len(a))
print("a 的第一个分量：", a[0])
print("向量 b：", b)

向量 a： [1, 2, 3]
a 的维度： 3
a 的第一个分量： 1
向量 b： [-1, 0, 2]


## 2. 向量加法与数乘

**逐元素相加**：`[a₁, a₂, a₃] + [b₁, b₂, b₃]` 应得到 `[a₁+b₁, a₂+b₂, a₃+b₃]`。

注意 Python 列表的 `a + b` **不是数学向量加法**，而是拼接列表。下面先观察这个区别。

In [2]:
print("列表 + 的结果：", a + b)

added = []
for index in range(len(a)):
    added.append(a[index] + b[index])
print("数学向量加法：", added)

scaled = []
for number in a:
    scaled.append(number * 2)
print("向量数乘 2：", scaled)

列表 + 的结果： [1, 2, 3, -1, 0, 2]
数学向量加法： [0, 2, 5]
向量数乘 2： [2, 4, 6]


将这些操作写成函数，让不同输入都能复用。两个向量做逐元素相加时需要**长度相等**，否则应明确报错。

In [3]:
def add_vectors(a, b):
    if len(a) != len(b):
        raise ValueError("向量加法要求两个向量长度相同")
    result = []
    for index in range(len(a)):
        result.append(a[index] + b[index])
    return result


def scale_vector(vector, factor):
    result = []
    for number in vector:
        result.append(number * factor)
    return result

print(add_vectors([1, 2], [3, 4]))  # [4, 6]
print(scale_vector([1, -2], 3))   # [3, -6]

[4, 6]
[3, -6]


## 3. 点积：对应相乘，再求和

对于三维向量：

\[
a\cdot b = a_1b_1 + a_2b_2 + a_3b_3
\]

例如，`a = [1, 2, 3]`，`b = [-1, 0, 2]`：

\[
1\times(-1)+2\times 0+3\times 2 = 5
\]

这不是把两个向量拼起来，而是计算**一个数**。如果长度不同，点积在本课的约定下不允许计算。

In [4]:
def dot(a, b):
    if len(a) != len(b):
        raise ValueError("点积要求两个向量长度相同")
    total = 0
    for index in range(len(a)):
        total += a[index] * b[index]
    return total

print("a · b =", dot(a, b))
assert dot(a, b) == 5
assert dot([1, 0], [0, 1]) == 0

a · b = 5


上一课学过：异常也需要验证。运行下面的单元，观察两个向量长度不同的处理。

In [5]:
try:
    print(dot([1, 2, 3], [4, 5]))
except ValueError as error:
    print("发现了维度问题：", error)

发现了维度问题： 点积要求两个向量长度相同


### 延伸：向量长度与余弦相似度

向量的欧氏长度（范数）是：

$
\lVert a\rVert = \sqrt{a\cdot a}
$

两个非零向量的余弦相似度是：

$
\cos(a,b)=\frac{a\cdot b}{\lVert a\rVert\lVert b\rVert}
$

它反映方向上的相似程度；对零向量不能使用这个公式。**点积不是余弦相似度**，不要混淆。

In [6]:
import math


def norm(vector):
    return math.sqrt(dot(vector, vector))


def cosine_similarity(a, b):
    denominator = norm(a) * norm(b)
    if denominator == 0:
        raise ValueError("零向量无法计算余弦相似度")
    return dot(a, b) / denominator

print("[3,4] 的长度：", norm([3, 4]))
print("a 与 b 的余弦相似度（约）：", round(cosine_similarity(a, b), 4))

[3,4] 的长度： 5.0
a 与 b 的余弦相似度（约）： 0.5976


## 4. 矩阵形状：先数行，再数列

二维列表可以表示矩阵：

```python
matrix = [
    [1, 2, 0],
    [0, 1, 3],
]
```

它有 **2 行、3 列**，所以形状是 `(2, 3)`。

> 本课约定矩阵至少有一行、一列，且每一行长度相同。`[[1, 2], [3]]` 不是本课允许的规则矩阵。

In [7]:
matrix = [[1, 2, 0], [0, 1, 3]]
print("行数：", len(matrix))
print("列数：", len(matrix[0]))
print("第 2 行第 3 列：", matrix[1][2])

行数： 2
列数： 3
第 2 行第 3 列： 3


In [8]:
def matrix_shape(matrix):
    if not matrix:
        raise ValueError("矩阵必须至少有一行")
    columns = len(matrix[0])
    if columns == 0:
        raise ValueError("矩阵每行必须至少有一个元素")
    for row in matrix:
        if len(row) != columns:
            raise ValueError("矩阵各行长度必须一致")
    return len(matrix), columns

print("矩阵形状：", matrix_shape(matrix))
assert matrix_shape(matrix) == (2, 3)

try:
    matrix_shape([[1, 2], [3]])
except ValueError as error:
    print("成功发现不规则矩阵：", error)

矩阵形状： (2, 3)
成功发现不规则矩阵： 矩阵各行长度必须一致


## 5. 矩阵乘向量：每一行做一次点积

`[2, 3]` 的矩阵乘 `[3]` 的向量，结果是一个有 **2** 个分量的向量。这里的 `[3]` 表示维度，而不是某个数值。

\[
\begin{bmatrix}
1&2&0\\
0&1&3
\end{bmatrix}
\begin{bmatrix}2\\1\\1\end{bmatrix}
=\begin{bmatrix}4\\4\end{bmatrix}
\]

第一行点积 `1×2 + 2×1 + 0×1 = 4`，第二行点积 `0×2 + 1×1 + 3×1 = 4`。

In [9]:
def matrix_vector(matrix, vector):
    _, columns = matrix_shape(matrix)
    if columns != len(vector):
        raise ValueError("矩阵列数必须等于向量长度")
    result = []
    for row in matrix:
        result.append(dot(row, vector))
    return result

x = [2, 1, 1]
print("矩阵形状：", matrix_shape(matrix))
print("输入向量长度：", len(x))
print("运算结果：", matrix_vector(matrix, x))
assert matrix_vector(matrix, x) == [4, 4]

矩阵形状： (2, 3)
输入向量长度： 3
运算结果： [4, 4]


## 6. 连接 Transformer：计算简化的 QKᵀ 分数

注意力中会使用 `Q`（查询向量）与 `K`（键向量）计算匹配分数。

假设一共有 3 个位置，每个位置的向量有 2 个数字：

```text
Q: 3 行 × 2 列
K: 3 行 × 2 列
```

对每个 Q 行向量，都与每个 K 行向量做一次点积，就得到 `3 × 3` 个分数；这相当于矩阵乘法 `QKᵀ`。

在标准缩放点积注意力中，还要把分数除以 `√dₖ`。本例 `dₖ=2`。

**特别注意：现在只得到 softmax 之前的分数，没有进行 softmax、未来掩码或 V 向量加权，因此还不是完整注意力机制。**

In [10]:
def qk_scaled_scores(queries, keys):
    _, query_width = matrix_shape(queries)
    _, key_width = matrix_shape(keys)
    if query_width != key_width:
        raise ValueError("Q 与 K 的每行向量维度必须相同")
    scores = []
    for query in queries:
        one_row = []
        for key in keys:
            one_row.append(dot(query, key) / math.sqrt(query_width))
        scores.append(one_row)
    return scores

Q = [[1, 0], [0, 1], [1, 1]]
K = [[1, 0], [0, 1], [1, 1]]
scores = qk_scaled_scores(Q, K)
print("Q 的形状：", matrix_shape(Q))
print("K 的形状：", matrix_shape(K))
print("QKᵀ / √dₖ 的形状：", matrix_shape(scores))
for row in scores:
    print([round(value, 4) for value in row])

Q 的形状： (3, 2)
K 的形状： (3, 2)
QKᵀ / √dₖ 的形状： (3, 3)
[0.7071, 0.0, 0.7071]
[0.0, 0.7071, 0.7071]
[0.7071, 0.7071, 1.4142]


### 观察形状的变化

让 Q 有 2 行、K 有 3 行，但每行仍保持 2 维。输出形状应从 `(3, 3)` 变成 `(2, 3)`。

矩阵形状规律：

\[
Q:[T_q,d_k],\quad K:[T_k,d_k]
\quad\Longrightarrow\quad QK^\top:[T_q,T_k]
\]

此时 `[T_q, T_k]` 的每个元素代表一个查询与一个键之间的匹配分数。

In [11]:
Q2 = [[1, 0], [0, 1]]
K2 = [[1, 0], [0, 1], [1, 1]]
scores2 = qk_scaled_scores(Q2, K2)
print("Q2：", matrix_shape(Q2), "K2：", matrix_shape(K2))
print("分数矩阵：", matrix_shape(scores2))
assert matrix_shape(scores2) == (2, 3)

Q2： (2, 2) K2： (3, 2)
分数矩阵： (2, 3)


## 7. 常见错误：先预测，再看结果

| 容易混淆的表达式 | 实际含义 |
|---|---|
| `a + b`（Python 列表） | 拼接列表，不是向量加法 |
| `a[0]` | 第一个分量，不是向量的长度 |
| `(2, 3)` | 2 行 3 列，不是 2 维向量 |
| `dot(a, b)` | 得到一个标量，不是一个新向量 |
| `QKᵀ / √dₖ` | 未经 softmax 的分数，还不是注意力概率 |

`zip(a, b)` 只遍历到较短序列结束。如果未经长度检查直接使用它实现点积，可能悄悄遗漏后面的分量。因此我们在 `dot()` 中先检查长度。

In [12]:
print("列表拼接：", [1, 2] + [3, 4])
print("真正的向量相加：", add_vectors([1, 2], [3, 4]))
print("点积：", dot([1, 2], [3, 4]))

列表拼接： [1, 2, 3, 4]
真正的向量相加： [4, 6]
点积： 11


## 8. 自己动手做（先不要往下看答案）

**练习 A｜手算与核对**

已知 `u=[2, -1, 3]`，`v=[1, 4, 0]`。先在纸上写出：

1. `u + v` 是什么？
2. `u · v` 是多少？
3. 矩阵 `[[2, 1], [0, -1], [3, 2]]` 的形状是什么？

**练习 B｜实现垂直判断**

写 `is_orthogonal(u, v)`：当两个等长向量点积等于 0 时返回 `True`，否则返回 `False`。**本练习只使用整数向量**；不同长度应沿用 `dot()` 的 `ValueError`。

**练习 C｜亲手实现未缩放的 QKᵀ**

写 `raw_qk_scores(Q, K)`：对每个查询和每个键做点积，返回二维列表，**不除以** `√dₖ`。输入不合法时使用前面写好的 `matrix_shape()` 检查。以下输入的结果应该是 `[[2, 1, 0], [1, 3, 1]]`。

```python
Q = [[1, 0], [0, 1]]
K = [[2, 1], [1, 3], [0, 1]]
```

### 练习区（请在下面单元中补全）

In [ ]:
# 练习 A：预测后再运行
u = [2, -1, 3]
v = [1, 4, 0]
print("u + v:", add_vectors(u, v))
print("u · v:", dot(u, v))
print("矩阵形状:", matrix_shape([[2, 1], [0, -1], [3, 2]]))

In [ ]:
# 练习 B：删除 pass，写出函数体后运行测试
# def is_orthogonal(u, v):
#     pass
#
# assert is_orthogonal([1, 0], [0, 1]) is True
# assert is_orthogonal([1, 2], [2, 1]) is False

In [ ]:
# 练习 C：删除 pass，写出函数体后运行测试
# def raw_qk_scores(Q, K):
#     pass
#
# assert raw_qk_scores([[1, 0], [0, 1]], [[2, 1], [1, 3], [0, 1]]) == [[2, 1, 0], [1, 3, 1]]

## 9. 参考答案（完成后再打开）

<details><summary>点击展开练习 A / B / C 参考思路</summary>

- **A：**`u + v = [3, 3, 3]`，`u · v = -2`，形状 `(3, 2)`。
- **B：**直接复用 `dot(u, v) == 0`，不要重复实现点积。
- **C：**先验证 Q 和 K 的每行宽度相同，再使用两层循环：外层遍历查询，内层遍历键，把每个 `dot(query, key)` 保存到对应结果行。

</details>

In [13]:
# 练习 B 参考答案
def is_orthogonal(u, v):
    return dot(u, v) == 0

assert is_orthogonal([1, 0], [0, 1]) is True
assert is_orthogonal([1, 2], [2, 1]) is False
print("练习 B 通过")

练习 B 通过


In [14]:
# 练习 C 参考答案
def raw_qk_scores(Q, K):
    _, q_width = matrix_shape(Q)
    _, k_width = matrix_shape(K)
    if q_width != k_width:
        raise ValueError("Q 与 K 的向量维度不一致")
    result = []
    for query in Q:
        row = []
        for key in K:
            row.append(dot(query, key))
        result.append(row)
    return result

assert raw_qk_scores(
    [[1, 0], [0, 1]],
    [[2, 1], [1, 3], [0, 1]],
) == [[2, 1, 0], [1, 3, 1]]
print("练习 C 通过")

练习 C 通过


## 10. 本课独立脚本与验收

同目录下的 **`lesson_08_vectors_dot_shapes.py`** 是完整可运行版本，包含与本讲义相同的核心函数、演示数据，以及**16 项固定检查**。

在终端运行：

```bash
python lesson_08_vectors_dot_shapes.py
```

你已经掌握本课的标准：

- 能手算三个元素的点积；能判断两个向量长度是否匹配。
- 能解释 `(2, 3)` 矩阵乘长度为 `3` 的向量为什么得到长度为 `2` 的结果。
- 能在不运行代码时预测 `Q:[2, 4]` 与 `K:[5, 4]` 计算 `QKᵀ` 后的形状 `(2, 5)`。
- 能区分缩放点积分数与 softmax 后的概率。

**下一课预告：第 09 课《NumPy 与张量基础》**。把今天用循环写出的点积和矩阵运算交给 NumPy，理解 `shape`、`dtype`、矩阵乘法以及为什么要使用数组运算。

### 完整脚本（与附带的 `.py` 文件相同）
下面这一个单元包含可独立运行的全部代码；教学过程中学过的函数会在这里重新定义一次，不会影响结果。

In [15]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第 08 课：向量、点积与矩阵形状。

运行：python lesson_08_vectors_dot_shapes.py
依赖：仅 Python 标准库，无需显卡、PyTorch 或 NumPy。
"""

import math


def add_vectors(a, b):
    """逐元素相加，要求两个向量等长。"""
    if len(a) != len(b):
        raise ValueError("向量加法要求两个向量长度相同")
    result = []
    for index in range(len(a)):
        result.append(a[index] + b[index])
    return result


def scale_vector(vector, factor):
    """向量数乘。"""
    result = []
    for number in vector:
        result.append(number * factor)
    return result


def dot(a, b):
    """点积：对应位置相乘后相加。"""
    if len(a) != len(b):
        raise ValueError("点积要求两个向量长度相同")
    total = 0
    for index in range(len(a)):
        total += a[index] * b[index]
    return total


def norm(vector):
    """向量的欧氏长度。"""
    return math.sqrt(dot(vector, vector))


def cosine_similarity(a, b):
    """余弦相似度；零向量没有定义。"""
    denominator = norm(a) * norm(b)
    if denominator == 0:
        raise ValueError("零向量无法计算余弦相似度")
    return dot(a, b) / denominator


def matrix_shape(matrix):
    """返回二维列表的形状 (行数, 列数)，拒绝空矩阵与不规则行。"""
    if not matrix:
        raise ValueError("矩阵必须至少有一行")
    columns = len(matrix[0])
    if columns == 0:
        raise ValueError("矩阵每行必须至少有一个元素")
    for row in matrix:
        if len(row) != columns:
            raise ValueError("矩阵各行长度必须一致")
    return len(matrix), columns


def matrix_vector(matrix, vector):
    """矩阵乘向量：每行与 vector 做一次点积。"""
    _, columns = matrix_shape(matrix)
    if columns != len(vector):
        raise ValueError("矩阵列数必须等于向量长度")
    result = []
    for row in matrix:
        result.append(dot(row, vector))
    return result


def qk_scaled_scores(queries, keys):
    """计算 QK^T/sqrt(d_k)，只返回未做 softmax 的分数矩阵。

    queries: [查询个数, d_k]，keys: [键的个数, d_k]
    输出: [查询个数, 键的个数]
    """
    _, query_width = matrix_shape(queries)
    _, key_width = matrix_shape(keys)
    if query_width != key_width:
        raise ValueError("Q 与 K 的每行向量维度必须相同")
    scores = []
    for query in queries:
        one_row = []
        for key in keys:
            one_row.append(dot(query, key) / math.sqrt(query_width))
        scores.append(one_row)
    return scores


def run_checks():
    """固定输入和预期结果，检查计算函数与维度错误处理。"""
    a = [1, 2, 3]
    b = [-1, 0, 2]
    matrix = [[1, 2, 0], [0, 1, 3]]
    q = [[1, 0], [0, 1], [1, 1]]
    k = [[1, 0], [0, 1], [1, 1]]
    scores = qk_scaled_scores(q, k)

    checks = [
        ("向量加法", add_vectors(a, b) == [0, 2, 5]),
        ("向量数乘", scale_vector(a, 2) == [2, 4, 6]),
        ("向量点积", dot(a, b) == 5),
        ("点积对称性", dot(a, b) == dot(b, a)),
        ("向量长度", math.isclose(norm([3, 4]), 5)),
        ("垂直向量余弦相似度", math.isclose(cosine_similarity([1, 0], [0, 1]), 0)),
        ("矩阵形状", matrix_shape(matrix) == (2, 3)),
        ("矩阵乘向量", matrix_vector(matrix, [2, 1, 1]) == [4, 4]),
        ("QK 分数矩阵形状", matrix_shape(scores) == (3, 3)),
        ("QK 第一项的缩放", math.isclose(scores[0][0], 1 / math.sqrt(2))),
        ("QK 末项的缩放", math.isclose(scores[2][2], math.sqrt(2))),
    ]
    passed = 0
    for label, okay in checks:
        if not okay:
            raise AssertionError("检查失败：" + label)
        passed += 1

    # 这些错误输入必须抛出 ValueError，而不能悄悄计算出一个结果。
    error_cases = [
        ("点积维度不匹配", lambda: dot([1, 2], [1])),
        ("矩阵行长不一致", lambda: matrix_shape([[1, 2], [3]])),
        ("矩阵乘向量维度不匹配", lambda: matrix_vector(matrix, [1, 2])),
        ("QK 维度不匹配", lambda: qk_scaled_scores([[1, 0]], [[1, 0, 2]])),
        ("零向量余弦相似度", lambda: cosine_similarity([0, 0], [1, 0])),
    ]
    for label, operation in error_cases:
        try:
            operation()
        except ValueError:
            passed += 1
        else:
            raise AssertionError("应当抛出 ValueError：" + label)

    print("通过固定检查：", passed, "项")
    return passed


def main():
    a = [1, 2, 3]
    b = [-1, 0, 2]
    print("向量 a：", a)
    print("向量 b：", b)
    print("a + b：", add_vectors(a, b))
    print("2 * a：", scale_vector(a, 2))
    print("a · b：", dot(a, b))
    print("a 的长度（约）：", round(norm(a), 4))
    print("余弦相似度（约）：", round(cosine_similarity(a, b), 4))

    matrix = [[1, 2, 0], [0, 1, 3]]
    vector = [2, 1, 1]
    print("矩阵形状：", matrix_shape(matrix))
    print("矩阵乘向量：", matrix_vector(matrix, vector))

    queries = [[1, 0], [0, 1], [1, 1]]
    keys = [[1, 0], [0, 1], [1, 1]]
    scores = qk_scaled_scores(queries, keys)
    print("Q 的形状：", matrix_shape(queries))
    print("K 的形状：", matrix_shape(keys))
    print("缩放点积分数的形状：", matrix_shape(scores))
    print("缩放点积分数（保留四位小数）：")
    for row in scores:
        rounded = []
        for number in row:
            rounded.append(round(number, 4))
        print(" ", rounded)
    print("注意：这些是 softmax 之前的分数，不是注意力概率。")
    run_checks()


if __name__ == "__main__":
    main()

向量 a： [1, 2, 3]
向量 b： [-1, 0, 2]
a + b： [0, 2, 5]
2 * a： [2, 4, 6]
a · b： 5
a 的长度（约）： 3.7417
余弦相似度（约）： 0.5976
矩阵形状： (2, 3)
矩阵乘向量： [4, 4]
Q 的形状： (3, 2)
K 的形状： (3, 2)
缩放点积分数的形状： (3, 3)
缩放点积分数（保留四位小数）：
  [0.7071, 0.0, 0.7071]
  [0.0, 0.7071, 0.7071]
  [0.7071, 0.7071, 1.4142]
注意：这些是 softmax 之前的分数，不是注意力概率。
通过固定检查： 16 项
